In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
# TODO: Read image data, feed into Claude
with open("pdf/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(messages, [
    # Image Block
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        }
    },
    # Text Block
    {
        "type": "text",
        "text": "Summarize the content of this PDF in one sentence."
    }
])

chat(messages)

Message(id='msg_01K4c5HSK6DSbNVaZ1myq8j7', container=None, content=[TextBlock(citations=None, text="This appears to be an aerial/satellite view of a residential property surrounded by dense vegetation. The image shows:\n\n1. **A house with a gray roof** - The building has an irregular or L-shaped footprint with what appears to be multiple sections or wings\n\n2. **Dense tree coverage** - The property is heavily wooded with thick forest or mature trees surrounding the structure on all sides\n\n3. **Natural setting** - The home appears to be in a secluded, wooded area with abundant green canopy coverage\n\n4. **Limited cleared area** - There's minimal clearing around the house, suggesting a rural or heavily forested residential setting\n\nThe overhead perspective and image quality suggest this is likely from satellite imagery or aerial photography, possibly from a mapping service. The property appears quite private and nestled within nature.", type='text')], model='claude-sonnet-4-5-2025